# Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, make_scorer

In [ ]:
SEED = 42

Import Preprocessed Train and Test Datasets

In [ ]:
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')

# Flattening of datasets for models
y_train_flat = y_train.values.ravel()
y_test_flat = y_test.values.ravel()

# Models with Default Hyperparameters

Setup of Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state = SEED),
    "SGD Classifier": SGDClassifier(random_state=SEED),
    "KNN": KNeighborsClassifier(),
    "Base Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "AdaBoost": AdaBoostClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "SVM": SVC(random_state=SEED)
}

Creation of False Positive Rate Function and Scorer

In [ ]:
def false_positive_rate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn)

fpr_scorer = make_scorer(false_positive_rate, greater_is_better=False)

Running of untuned models

In [ ]:
fpr_cv_scores = {}
fpr_train_scores = {}
fpr_test_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train_flat)

    # In-sample FPR
    y_pred_train = model.predict(X_train)
    fpr_train_scores[name] = false_positive_rate(y_train_flat, y_pred_train)

    # 5-fold Cross-validation to determine overfitting
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

    scores = - cross_val_score(model, X_train, y_train_flat, cv=kf, scoring=fpr_scorer)
    fpr_cv_scores[name] = scores

    # Out-of-sample FPR
    y_pred_test = model.predict(X_test)
    fpr_test_scores[name] = false_positive_rate(y_test_flat, y_pred_test)

## Presentation of Metrics

In [ ]:
results_df = pd.DataFrame({
    'Model': models.keys(),
    'In-Sample FPR': fpr_train_scores.values(),
    'Mean CV FPR': [scores.mean() for scores in fpr_cv_scores.values()],
    'Out-of-Sample FPR': fpr_test_scores.values()
    })

results_df

Test FPR Visualizations

In [ ]:
model_medians = []
for name, scores in fpr_cv_scores.items():
    model_medians.append((np.median(scores), name, scores))

model_medians.sort(key=lambda x: x[0])

boxplot_data = [item[2] for item in model_medians]
boxplot_labels = [item[1] for item in model_medians]

In [ ]:
# Prepare data for boxplot
plt.boxplot(boxplot_data, labels=boxplot_labels, patch_artist=True)
plt.title('Cross-Validation False Positive Rates for Base Models')
plt.xlabel('Model')
plt.ylabel('False Positive Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
sorted_test_df = results_df.sort_values('Out-of-Sample FPR', ascending = False)
plt.barh(sorted_test_df['Model'], sorted_test_df['Out-of-Sample FPR'])
plt.title('FPR for Base Classifiers on Test Set')
plt.xlabel('False Positive Rate')
plt.ylabel('Model')
plt.show()

# Hyperparameter Tuning

## Logistic Regression

In [ ]:
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

lr_model = LogisticRegression(max_iter=1000, random_state=SEED)

random_search_lr = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=param_grid_lr,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_lr.fit(X_train, y_train_flat)
best_lr_model = random_search_lr.best_estimator_
y_pred_lr_tuned = best_lr_model.predict(X_test)

print("\nBest parameters found for Logistic Regression:", random_search_lr.best_params_)
print("Best cross-validation FPR:", -random_search_lr.best_score_)
print("\nFPR for Tuned Logistic Regression on Test Set:", false_positive_rate(y_test_flat, y_pred_lr_tuned))

### SGD Classifier

In [ ]:
param_grid_sgd = {
    'loss': ['hinge', 'log_loss'],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'alpha': [0.0001, 0.001, 0.01]
}

sgd_model = SGDClassifier(random_state = SEED)

random_search_sgd = RandomizedSearchCV(
    estimator=sgd_model,
    param_distributions=param_grid_sgd,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state = SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_sgd.fit(X_train, y_train_flat)
best_sgd_model = random_search_sgd.best_estimator_
y_pred_sgd_tuned = best_sgd_model.predict(X_test)

print("\nBest parameters found for SGD Classifier:", random_search_sgd.best_params_)
print("Best cross-validation FPR:", random_search_sgd.best_score_)
print("\nFPR for Tuned SGD Classifier on Test Set:", false_positive_rate(y_test_flat, y_pred_sgd_tuned))

## KNN

In [ ]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_model = KNeighborsClassifier()

random_search_knn = RandomizedSearchCV(
    estimator=knn_model,
    param_distributions=param_grid_knn,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state = SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_knn.fit(X_train, y_train_flat)
best_knn_model = random_search_knn.best_estimator_
y_pred_knn_tuned = best_knn_model.predict(X_test)

print("\nBest parameters found for K-Nearest Neighbors:", random_search_knn.best_params_)
print("Best cross-validation FPR:", random_search_knn.best_score_)
print("\nFPR for Tuned K-Nearest Neighbors on Test Set:", false_positive_rate(y_test_flat, y_pred_knn_tuned))

### Base Decision Tree

In [ ]:
param_grid_dt = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_model = DecisionTreeClassifier(random_state = SEED)

random_search_dt = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=param_grid_dt,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state = SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_dt.fit(X_train, y_train_flat)
best_dt_model = random_search_dt.best_estimator_
y_pred_dt_tuned = best_dt_model.predict(X_test)

print("\nBest parameters found for Decision Tree:", random_search_dt.best_params_)
print("Best cross-validation FPR:", random_search_dt.best_score_)
print("\nFPR for Tuned Decision Tree on Test Set:", false_positive_rate(y_test_flat, y_pred_dt_tuned))

## AdaBoost Decision Trees

In [ ]:
param_grid_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1]
}

ada_model = AdaBoostClassifier(random_state = SEED)

random_search_ada = RandomizedSearchCV(
    estimator=ada_model,
    param_distributions=param_grid_ada,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state = SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_ada.fit(X_train, y_train_flat)
best_ada_model = random_search_ada.best_estimator_
y_pred_ada_tuned = best_ada_model.predict(X_test)

print("\nBest parameters found for AdaBoost:", random_search_ada.best_params_)
print("Best cross-validation FPR:", random_search_ada.best_score_)
print("\nFPR for Tuned AdaBoost on Test Set:", false_positive_rate(y_test_flat, y_pred_ada_tuned))

## Random Forest

In [ ]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_model = RandomForestClassifier(random_state = SEED)

random_search_rf = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid_rf,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_rf.fit(X_train, y_train_flat)
best_rf_model = random_search_rf.best_estimator_
y_pred_rf_tuned = best_rf_model.predict(X_test)

print("\nBest parameters found for Random Forest:", random_search_rf.best_params_)
print("Best cross-validation FPR:", -random_search_rf.best_score_)
print("\nFPR for Tuned Random Forest on Test Set:", false_positive_rate(y_test_flat, y_pred_rf_tuned))

## Gradient Boosting

In [ ]:
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

gb_model = GradientBoostingClassifier(random_state = SEED)

random_search_gb = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_grid_gb,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state = SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_gb.fit(X_train, y_train_flat)
best_gb_model = random_search_gb.best_estimator_
y_pred_gb_tuned = best_gb_model.predict(X_test)

print("\nBest parameters found for Gradient Boosting:", random_search_gb.best_params_)
print("Best cross-validation FPR:", random_search_gb.best_score_)
print("\nFPR for Tuned Gradient Boosting on Test Set:", false_positive_rate(y_test_flat, y_pred_gb_tuned))

## SVM

In [ ]:
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

svm_model = SVC(random_state = SEED)

random_search_svm = RandomizedSearchCV(
    estimator=svm_model,
    param_distributions=param_grid_svm,
    scoring=fpr_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state = SEED),
    verbose=1,
    n_jobs=-1,
    random_state = SEED
)

random_search_svm.fit(X_train, y_train_flat)
best_svm_model = random_search_svm.best_estimator_
y_pred_svm_tuned = best_svm_model.predict(X_test)

print("\nBest parameters found for SVM:", random_search_svm.best_params_)
print("Best cross-validation FPR:", random_search_svm.best_score_)
print("\nFPR for Tuned SVM on Test Set:", false_positive_rate(y_test_flat, y_pred_svm_tuned))

## Voting Classifier (Ensemble of Tuned Models)

In [ ]:
estimators = [
    ('logistic_regression', best_lr_model),
    ('sgd_classifier', best_sgd_model),
    ('knn', best_knn_model),
    ('base_decision_tree', best_dt_model),
    ('adaboost', best_ada_model),
    ('random_forest', best_rf_model),
    ('gradient_boosting', best_gb_model),
    ('svm', best_svm_model)
]

voting_clf = VotingClassifier(estimators=estimators, voting='hard', n_jobs=-1, verbose=True)

voting_clf.fit(X_train, y_train_flat)
y_pred_voting = voting_clf.predict(X_test)

print("\nFPR for Voting Classifier on Test Set:", false_positive_rate(y_test_flat, y_pred_voting))


### FPR Performance Comparison

In [ ]:
model_fpr_performance = {
    'Logistic Regression': false_positive_rate(y_test_flat, y_pred_lr_tuned),
    'Random Forest': false_positive_rate(y_test_flat, y_pred_rf_tuned),
    'Gradient Boosting': false_positive_rate(y_test_flat, y_pred_gb_tuned),
    'SVM': false_positive_rate(y_test_flat, y_pred_svm_tuned),
    'SGD Classifier': false_positive_rate(y_test_flat, y_pred_sgd_tuned),
    'AdaBoost': false_positive_rate(y_test_flat, y_pred_ada_tuned),
    'Base Decision Tree': false_positive_rate(y_test_flat, y_pred_dt_tuned),
    'KNN': false_positive_rate(y_test_flat, y_pred_knn_tuned),
    'Voting Classifier': false_positive_rate(y_test_flat, y_pred_voting)
}

# Convert to DataFrame for easier plotting
fpr_df = pd.DataFrame(model_fpr_performance.items(), columns=['Model', 'FPR'])

# Sort by FPR for better readability
fpr_df = fpr_df.sort_values(by='FPR', ascending=False)

plt.barh(fpr_df['Model'], fpr_df['FPR'])
plt.title('FPR for Tuned Classifiers on Test Set')
plt.xlabel('False Positive Rate')
plt.ylabel('Model')
plt.tight_layout()
plt.show()

### Cross-Validation FPR Performance Comparison

In [ ]:
cv_fpr_performance = {
    'Logistic Regression': -random_search_lr.best_score_,
    'Random Forest': -random_search_rf.best_score_,
    'Gradient Boosting': -random_search_gb.best_score_,
    'SVM': -random_search_svm.best_score_,
    'SGD Classifier': -random_search_sgd.best_score_,
    'AdaBoost': -random_search_ada.best_score_,
    'Decision Tree': -random_search_dt.best_score_,
    'K-Nearest Neighbors': -random_search_knn.best_score_
}

cv_fpr_df = pd.DataFrame(cv_fpr_performance.items(), columns=['Model', 'CV_FPR'])

cv_fpr_df = cv_fpr_df.sort_values(by='CV_FPR', ascending=False)

plt.barh(cv_fpr_df['Model'], cv_fpr_df['CV_FPR'])
plt.title('Cross-Validation False Positive Rate (FPR) for Tuned Models')
plt.xlabel('Cross-Validation False Positive Rate (FPR)')
plt.ylabel('Model')
plt.tight_layout()
plt.show()

### Comparison of Base vs. Tuned Models FPR

In [ ]:
comparison_data = []

models_to_compare = [
    'Logistic Regression',
    'SGD Classifier',
    'KNN',
    'Base Decision Tree',
    'AdaBoost',
    'Random Forest',
    'Gradient Boosting',
    'SVM'
]

for model_name_untuned in models_to_compare:
    fpr_untuned = fpr_test_scores[model_name_untuned]

    fpr_tuned = model_fpr_performance.get(model_name_untuned)

    comparison_data.append({
        'Model': model_name_untuned,
        'FPR': fpr_untuned,
        'Type': 'Base'
    })
    comparison_data.append({
        'Model': model_name_untuned,
        'FPR': fpr_tuned,
        'Type': 'Tuned'
    })

fpr_comparison_df = pd.DataFrame(comparison_data)

sorted_models_by_untuned_fpr = fpr_comparison_df[fpr_comparison_df['Type'] == 'Base'].sort_values(by='FPR')['Model'].tolist()
fpr_comparison_df['Model'] = pd.Categorical(fpr_comparison_df['Model'], categories=sorted_models_by_untuned_fpr, ordered=True)
fpr_comparison_df = fpr_comparison_df.sort_values(['Model', 'Type'])

plt.barplot(x='FPR', y='Model', hue='Type', data=fpr_comparison_df)
plt.title('FPR Comparison of Base and Tuned Models on Test Set')
plt.xlabel('False Positive Rate')
plt.ylabel('Model')
plt.legend(title='Model Type')
plt.tight_layout()
plt.show()